Purpose
---
Split a single EU Fleet Register CSV export into one CSV file per country.

Input:
    *eufr_data.csv*

Output:
    *eufr_data_by_country/eufr_data_<COUNTRY>.csv*

The script expects a column named:
    *Country of Registration*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/BCKG/CSV Barcos"

Mounted at /content/drive
/content/drive/MyDrive/BCKG/CSV Barcos


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

INPUT_FILE = Path("eufr_data.csv")
OUTPUT_DIR = Path("eufr_data_by_country")
COUNTRY_COLUMN = "Country of Registration"
OUTPUT_PREFIX = "eufr_data"

def safe_filename(value: str) -> str:
    """
    Convert a country value into a safe filename fragment.

    This removes characters that are invalid on common filesystems and
    normalizes whitespace to underscores.
    """
    name = str(value).strip()
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name


def load_eufr_csv(path: Path) -> pd.DataFrame:
    """
    Load the EUFR CSV export as strings.

    Keeping every column as string avoids pandas converting identifiers,
    dates, registration numbers or leading-zero values.
    """
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    df = pd.read_csv(path, sep=";", dtype=str)

    if COUNTRY_COLUMN not in df.columns:
        raise ValueError(
            f"Missing required column '{COUNTRY_COLUMN}'. "
            f"Available columns: {list(df.columns)}"
        )

    return df


def split_by_country(df: pd.DataFrame, output_dir: Path) -> int:
    """
    Write one CSV file per country of registration.

    Rows with missing or empty country values are ignored.
    Returns the number of files generated.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    files_written = 0

    for country, group in df.groupby(COUNTRY_COLUMN, dropna=True):
        country_value = str(country).strip()
        if not country_value:
            continue

        filename = f"{OUTPUT_PREFIX}_{safe_filename(country_value)}.csv"
        filepath = output_dir / filename

        group.to_csv(filepath, sep=";", index=False, encoding="utf-8-sig")

        files_written += 1
        print(f"Saved: {filepath} ({len(group)} rows)")

    return files_written

def main() -> int:
    """
    Load the source EUFR CSV and generate country-level CSV files.
    """
    print(f"Input file: {INPUT_FILE}")
    print(f"Output directory: {OUTPUT_DIR}")

    df = load_eufr_csv(INPUT_FILE)
    print(f"Rows loaded: {len(df)}")

    files_written = split_by_country(df, OUTPUT_DIR)
    print(f"Files generated: {files_written}")

    return 0


if __name__ == "__main__":
    main()